In [0]:
INSERT INTO sales_catalog.gold.dim_customers (
    customer_key,
    customer_id,
    custmer_number,
    first_name,
    last_name,
    country,
    marital_status,
    gender,
    birthdate,
    create_date
)
SELECT 
    ROW_NUMBER() OVER(ORDER BY ci.cst_id) AS customer_key,
    ci.cst_id AS customer_id, 
    ci.cst_key AS custmer_number, 
    ci.cst_firstname AS first_name, 
    ci.cst_lastname AS last_name, 
    la.CNTRY AS country,
    ci.cst_marital_status AS marital_status, 
    CASE 
        WHEN ci.cst_gndr != 'Unknown' THEN ci.cst_gndr
        ELSE COALESCE(ca.gen, 'Unknown') 
    END AS gender,
    ca.BDATE AS birthdate,
    ci.cst_create_date AS create_date
   FROM sales_catalog.silver.s_crm_cust_info ci
LEFT JOIN sales_catalog.silver.s_cust_az12 ca
    ON ci.cst_key = ca.CID
LEFT JOIN sales_catalog.silver.s_loc_a101_cntry la
    ON ci.cst_key = la.CID

In [0]:
%sql
SELECT 
    ROW_NUMBER() OVER(ORDER BY ci.cst_id) AS customer_key,
    ci.cst_id AS customer_id, 
    ci.cst_key AS custmer_number, 
    ci.cst_firstname AS first_name, 
    ci.cst_lastname AS last_name, 
    la.CNTRY AS country,
    ci.cst_marital_status AS marital_status, 
    CASE 
        WHEN ci.cst_gndr != 'Unknown' THEN ci.cst_gndr
        ELSE COALESCE(ca.gen, 'Unknown') 
    END AS gender,
    ca.BDATE AS birthdate,
    ci.cst_create_date AS create_date
   FROM sales_catalog.silver.s_crm_cust_info ci
LEFT JOIN sales_catalog.silver.s_cust_az12 ca
    ON ci.cst_key = ca.CID
LEFT JOIN sales_catalog.silver.s_loc_a101_cntry la
    ON ci.cst_key = la.CID

LIMIT 3


In [0]:
%sql 
INSERT INTO sales_catalog.gold.dim_products (
    product_key,
    product_id,
    product_number,
    product_name,
    category_id,
    category,
    subcategory,
    maintenance,
    cost,
    product_line,
    start_date
)
SELECT 
    ROW_NUMBER() OVER(ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key,
    pn.prd_id AS product_id, 
    pn.prd_key AS product_number, 
    pn.prd_nm AS product_name, 
    pn.cat_id AS category_id, 
    pc.CAT AS category, 
    pc.SUBCAT AS subcategory, 
    pc.MAINTENANCE AS maintenance, 
    pn.prd_cost AS cost, 
    pn.prd_line AS product_line, 
    pn.prd_start_dt AS start_date
FROM sales_catalog.silver.s_crm_prd_info pn
LEFT JOIN sales_catalog.silver.s_px_cat_g1v2 pc
    ON pn.cat_id = pc.ID
WHERE pn.prd_end_dt IS NULL

In [0]:
%sql
select * from sales_catalog.gold.dim_products limit 5

In [0]:
INSERT INTO sales_catalog.gold.fact_sales_orders (
    order_number,
    product_key,
    customer_key,
    order_date,
    shipping_date,
    due_date,
    sales_amount,
    quantity,
    price
)
SELECT 
    sd.sls_ord_num AS order_number,
    pr.product_key,
    cu.customer_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt AS shipping_date,
    sd.sls_due_dt AS due_date,
    sd.sls_sales AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price AS price
FROM sales_catalog.silver.s_sales_details sd
LEFT JOIN sales_catalog.gold.dim_products pr 
ON sd.sls_prd_key = pr.product_number
LEFT JOIN sales_catalog.gold.dim_customers cu 
ON sd.sls_cust_id = cu.customer_id

In [0]:
%sql
select count(*) from sales_catalog.gold.fact_sales